In [5]:
# --- Imports ---
import pandas as pd, numpy as np
from itertools import combinations
import lang2vec.lang2vec as l2v
from cluster_lang2vec_distances import auto_k, build_distance_matrix, cluster_with_k

# --- Load + filter ---
df = pd.read_csv(
    "/home/ubuntu/dice_repos/HTYLLM-PG/approaches/CoLA/data_prep/base_data/fineweb2-language-distribution.csv"
)
df = df[(df.split == "train") & ~df.subset.str.contains("_removed$", na=False)]

top = (
    df.assign(documents=pd.to_numeric(df.documents, errors="coerce").fillna(0))
      .groupby(["code", "name", "family", "resource_availability"], as_index=False)
      ["documents"].sum()
      .sort_values("documents", ascending=False)
      .head(200)
)

langs = top[top.code.isin(l2v.DISTANCE_LANGUAGES)].reset_index(drop=True)
codes = langs.code.tolist()

# --- lang2vec distance matrix (NUMERIC) ---
D = build_distance_matrix(codes, "genetic")
dist = pd.DataFrame(D, index=codes, columns=codes)

# --- Family medoids ---
family_rep = {}
for fam, g in langs.groupby("family"):
    fam_codes = g.code.tolist()
    if len(fam_codes) < 3:
        continue
    family_rep[fam] = min(
        fam_codes,
        key=lambda c: dist.loc[c, fam_codes].mean()
    )

# --- Select 4 maximally distant families ---
fam_docs = langs.groupby("family")["documents"].sum().sort_values(ascending=False)
selected = [fam_docs.index[0]]

while len(selected) < 4:
    selected.append(
        max(
            (f for f in family_rep if f not in selected),
            key=lambda f: min(
                dist.loc[family_rep[f], family_rep[s]] for s in selected
            )
        )
    )

# --- Pick tightest triplet per family ---
experts = {}
for fam in selected:
    fam_codes = langs[langs.family == fam].code.tolist()
    best = min(
        combinations(fam_codes, 3),
        key=lambda trio: np.mean(
            [dist.loc[a, b] for a, b in combinations(trio, 2)]
        )
    )
    experts[fam] = best

print(experts)


ModuleNotFoundError: No module named 'cluster_lang2vec_distances'